# Data Preprocessing and Feature Engineering

Raw data is rarely ready for machine learning. Real-world datasets arrive with missing values, inconsistent formats, mixed data types, outliers, duplicate rows, and irrelevant columns. A model trained on messy data learns noise instead of signal — garbage in, garbage out.

**Data preprocessing** transforms raw data into a clean, consistent format that algorithms can process. **Feature engineering** goes further: it creates new input variables from existing ones to give the model more useful information. Together, these steps often determine whether a project succeeds or fails more than the choice of algorithm.

Experienced practitioners spend a large fraction of their time on data preparation. This notebook covers the essential techniques: cleaning data, handling missing values and outliers, encoding categories, scaling features, engineering new features, selecting the most informative ones, and building reproducible preprocessing pipelines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Preprocessing Matters

Machine learning algorithms make assumptions about the data they receive. Linear models assume features are on comparable scales. Distance-based algorithms (K-Nearest Neighbors, K-Means) are sensitive to feature magnitude — a feature measured in thousands dominates one measured in fractions. Neural networks converge faster when inputs are normalized. Tree-based models are less sensitive to scaling but still benefit from clean, well-encoded data.

Consider a dataset predicting loan approval with features: annual income (\$20,000–\$200,000), credit score (300–850), and employment status (employed, self-employed, unemployed). Income and credit score are numerical but on vastly different scales. Employment status is categorical text. Some rows may have missing credit scores. Without preprocessing, a distance-based model would treat income differences as far more important than credit score differences, and it cannot process text categories at all.

Preprocessing bridges the gap between raw human-readable data and the numerical matrices that algorithms require.

---

## 2. Exploring and Understanding Raw Data

Before transforming data, you must understand it. Exploratory analysis answers: What types are the columns? How many values are missing? What are the distributions? Are there duplicates or inconsistencies?

**Data types** fall into broad categories:

- **Numerical** — integers or floats (age, price, temperature).
- **Categorical** — discrete labels (color, country, product type).
- **Ordinal** — categories with meaningful order (low/medium/high, education level).
- **Text** — free-form strings (reviews, descriptions) requiring specialized processing.
- **Datetime** — timestamps that can be decomposed into year, month, day, hour.

**Common data quality issues:**

- Missing values (NaN, null, empty strings, "N/A").
- Outliers (values far from the typical range).
- Inconsistent formatting ("USA", "U.S.A.", "United States").
- Duplicate rows.
- Incorrect data types (numbers stored as strings).
- Irrelevant or redundant columns (IDs, constant columns).

In [ ]:
# Sample messy dataset
data = {
    "age": [25, 30, np.nan, 45, 22, 38, 55, 29],
    "income": [45000, 62000, 58000, 95000, 32000, 71000, 120000, 48000],
    "credit_score": [720, 680, np.nan, 810, 590, 740, 850, 700],
    "employment": ["employed", "self-employed", "employed", "employed",
                   "unemployed", "employed", "employed", "self-employed"],
    "approved": [1, 1, 0, 1, 0, 1, 1, 0]
}
df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nBasic statistics:\n", df.describe())
print("\nCategorical value counts:\n", df["employment"].value_counts())

---

## 3. Handling Missing Values

Missing data is ubiquitous. Sensors fail, users skip form fields, records are incomplete. Ignoring missing values crashes most algorithms; handling them thoughtfully is essential.

### 3.1 Types of Missingness

- **MCAR (Missing Completely At Random)** — Missingness is unrelated to any variable. Dropping or imputing does not introduce bias.
- **MAR (Missing At Random)** — Missingness depends on observed variables but not the missing value itself. Imputation using other features is reasonable.
- **MNAR (Missing Not At Random)** — Missingness depends on the missing value itself. Example: high-income individuals refuse to report income. This is the hardest case and may require domain-specific handling.

### 3.2 Strategies

**Deletion:**
- *Listwise deletion* — remove entire rows with any missing value. Simple but loses data.
- *Column deletion* — remove features with too many missing values (e.g., >50%).

**Imputation** — fill missing values with estimated ones:
- *Mean / median / mode* — for numerical and categorical features respectively.
- *Forward / backward fill* — for time series, use previous or next value.
- *Model-based imputation* — predict missing values using other features (KNN imputation, regression imputation).
- *Constant imputation* — fill with a sentinel value like -1 or "unknown".

The choice depends on how much data is missing and why. Median imputation is robust to outliers for numerical features. Never impute using statistics computed on the full dataset including the test set — that causes data leakage.

In [ ]:
# Missing value imputation with SimpleImputer
numerical_cols = ["age", "income", "credit_score"]

imputer = SimpleImputer(strategy="median")
df_imputed = df.copy()
df_imputed[numerical_cols] = imputer.fit_transform(df[numerical_cols])

print("Before imputation:")
print(df[numerical_cols].isnull().sum(), "missing values")
print("\nAfter median imputation:")
print(df_imputed[numerical_cols])
print("\nImputed values (medians used):", imputer.statistics_)

---

## 4. Handling Outliers

Outliers are data points that deviate significantly from the rest of the distribution. They may represent genuine rare events (a billionaire in an income dataset) or errors (a salary entered as 999999999).

### 4.1 Detection Methods

- **Z-score** — Flag points where $|z| = \frac{x - \mu}{\sigma} > 3$ (more than 3 standard deviations from the mean).
- **IQR method** — Compute the interquartile range $IQR = Q_3 - Q_1$. Flag points below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$.
- **Visualization** — Box plots and scatter plots reveal outliers visually.

### 4.2 Treatment

- **Remove** — Delete outlier rows if they are clearly errors.
- **Cap (winsorize)** — Replace extreme values with threshold bounds.
- **Transform** — Apply log or square root transformation to compress the range.
- **Keep** — If outliers are genuine and informative, retain them (tree models handle them well).

Blindly removing all outliers can discard valuable information. Always investigate whether they are errors or real phenomena before deciding.

In [ ]:
# Outlier detection with IQR method
incomes = np.array([32000, 45000, 48000, 58000, 62000, 71000, 95000, 120000, 5000000])

Q1, Q3 = np.percentile(incomes, [25, 75])
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = incomes[(incomes < lower) | (incomes > upper)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(incomes, vert=True)
axes[0].set_title("Box plot reveals outlier")
axes[0].set_ylabel("Income")

axes[1].hist(incomes, bins=8, edgecolor="black", alpha=0.7)
axes[1].axvline(upper, color="r", linestyle="--", label=f"Upper bound: {upper:,.0f}")
axes[1].set_title("Histogram with IQR upper bound")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"IQR bounds: [{lower:,.0f}, {upper:,.0f}]")
print(f"Outliers detected: {outliers}")

---

## 5. Encoding Categorical Variables

Machine learning algorithms require numerical input. Categorical variables — text labels like "employed" or colors like "red" — must be converted to numbers.

### 5.1 Label Encoding

Assigns each category a unique integer: employed=0, self-employed=1, unemployed=2. Simple and memory-efficient, but it **implies an order** that may not exist. A model might incorrectly assume unemployed (2) > employed (0).

Best for: **ordinal** variables where order matters (low=0, medium=1, high=2) and **tree-based models** that can split on categorical values without assuming order.

### 5.2 One-Hot Encoding

Creates a binary column for each category. "employed" becomes [1, 0, 0], "self-employed" becomes [0, 1, 0], "unemployed" becomes [0, 0, 1]. No false ordering is imposed.

Best for: **nominal** variables with no inherent order (color, city, product type). Used with linear models, neural networks, and distance-based algorithms.

Drawback: High-cardinality features (thousands of categories) create thousands of columns. Techniques like target encoding or embedding layers address this.

### 5.3 Ordinal Encoding

Like label encoding but with explicitly defined order. Education: high school=0, bachelor=1, master=2, PhD=3. The integer values reflect the true ordering.

In [ ]:
categories = ["employed", "self-employed", "unemployed", "employed", "self-employed"]

# Label encoding
le = LabelEncoder()
label_encoded = le.fit_transform(categories)
print("Label encoding:")
for cat, code in zip(categories, label_encoded):
    print(f"  {cat:15s} → {code}")

# One-hot encoding
ohe = OneHotEncoder(sparse_output=False)
one_hot = ohe.fit_transform(np.array(categories).reshape(-1, 1))
ohe_df = pd.DataFrame(one_hot, columns=ohe.get_feature_names_out(["employment"]))
print("\nOne-hot encoding:")
print(ohe_df.to_string(index=False))

---

## 6. Feature Scaling

Features measured on different scales can distort model behavior. Scaling puts features on a comparable range.

### 6.1 Standardization (Z-score Normalization)

Transforms each feature to have mean 0 and standard deviation 1:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

Result: most values fall between -3 and +3. Used when features are approximately normally distributed. Required for algorithms sensitive to scale: logistic regression, SVM, KNN, neural networks, PCA.

### 6.2 Min-Max Normalization

Scales features to a fixed range, typically [0, 1]:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Preserves the shape of the distribution. Sensitive to outliers — a single extreme value compresses all other values into a narrow range.

### 6.3 When Not to Scale

Tree-based models (decision trees, random forests, gradient boosting) split on individual features and are **invariant to monotonic transformations**. Scaling does not change their predictions. However, scaling is still useful when combining tree models with other algorithms in an ensemble or pipeline.

In [ ]:
features = df_imputed[["age", "income", "credit_score"]].values

scaler_std = StandardScaler()
scaler_mm = MinMaxScaler()

scaled_std = scaler_std.fit_transform(features)
scaled_mm = scaler_mm.fit_transform(features)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].boxplot(features, labels=["age", "income", "credit"])
axes[0].set_title("Original (different scales)")

axes[1].boxplot(scaled_std, labels=["age", "income", "credit"])
axes[1].set_title("Standardized (mean=0, std=1)")

axes[2].boxplot(scaled_mm, labels=["age", "income", "credit"])
axes[2].set_title("Min-Max scaled [0, 1]")

plt.tight_layout()
plt.show()

---

## 7. Feature Engineering

Feature engineering is the art of creating new input variables from existing data to help the model learn better patterns. Good features can make a simple model outperform a complex model on poor features.

### 7.1 Mathematical Transformations

- **Polynomial features** — Create $x^2$, $x_1 \cdot x_2$ to capture nonlinear relationships. A linear model with polynomial features can fit curves.
- **Log transform** — $\log(x)$ compresses right-skewed distributions (income, population).
- **Binning** — Convert continuous values into discrete buckets (age groups: 18–25, 26–35, etc.).

### 7.2 Domain-Driven Features

The most powerful features come from domain knowledge:

- **Finance:** debt-to-income ratio, loan-to-value ratio.
- **E-commerce:** days since last purchase, average order value, purchase frequency.
- **Healthcare:** body mass index (weight / height²), years since diagnosis.

### 7.3 Datetime Features

A single timestamp contains rich information when decomposed:

- Year, month, day of week, hour, quarter.
- Is weekend? Is holiday? Is business hours?
- Days since a reference event.

### 7.4 Text-Derived Features

From text columns: word count, character count, presence of keywords, sentiment score. More advanced text processing uses tokenization, n-grams, and learned embeddings to represent language numerically.

### 7.5 Aggregation Features

Group-level statistics: average purchase amount per customer, number of transactions per category, rolling 7-day mean of sensor readings.

In [ ]:
# Feature engineering examples
df_eng = df_imputed.copy()

# Domain feature: debt-to-income ratio (simulated debt)
debt = np.array([15000, 22000, 18000, 35000, 25000, 20000, 10000, 16000])
df_eng["debt_to_income"] = debt / df_eng["income"]

# Binning: age groups
df_eng["age_group"] = pd.cut(df_eng["age"], bins=[0, 30, 45, 100],
                              labels=["young", "middle", "senior"])

# Polynomial: interaction between income and credit score
df_eng["income_x_credit"] = df_eng["income"] * df_eng["credit_score"] / 1e6

# Datetime decomposition
dates = pd.to_datetime(["2024-01-15", "2024-06-20", "2024-03-10", "2024-12-01",
                       "2024-08-05", "2024-02-28", "2024-11-11", "2024-07-04"])
df_eng["application_month"] = dates.month
df_eng["is_q4"] = dates.quarter == 4

print("Engineered features:")
print(df_eng[["age", "debt_to_income", "age_group", "income_x_credit",
              "application_month", "is_q4"]].to_string())

---

## 8. Feature Selection

Not all features help prediction. Irrelevant or redundant features add noise, increase training time, and risk overfitting. Feature selection identifies the most informative inputs.

### 8.1 Filter Methods

Evaluate each feature independently using a statistical test, then keep the top-K:

- **Correlation** — Remove features highly correlated with each other (redundancy).
- **Chi-squared test** — For categorical features vs categorical target.
- **ANOVA F-test** — For numerical features vs categorical target.
- **Mutual information** — Measures dependency between feature and target.

Fast and model-agnostic, but ignores feature interactions.

### 8.2 Wrapper Methods

Use the model itself to evaluate feature subsets:

- **Forward selection** — Start empty, add one feature at a time.
- **Backward elimination** — Start with all, remove one at a time.
- **Recursive Feature Elimination (RFE)** — Train model, remove least important feature, repeat.

More accurate but computationally expensive.

### 8.3 Embedded Methods

Feature selection built into the training process:

- **Lasso (L1) regression** — Shrinks some coefficients to exactly zero, effectively removing features.
- **Tree feature importance** — Random forests and gradient boosting report which features contributed most to splits.

### 8.4 Dimensionality Reduction

Rather than selecting individual features, project all features into a lower-dimensional space. PCA creates new composite features that capture maximum variance. Useful when many features are correlated.

In [ ]:
# Feature selection with ANOVA F-test
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=20, n_informative=5,
                            n_redundant=5, random_state=42)

selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X, y)

scores = selector.scores_
top_features = np.argsort(scores)[::-1][:5]

plt.figure(figsize=(10, 4))
plt.bar(range(20), scores, color="steelblue", alpha=0.7)
plt.bar(top_features, scores[top_features], color="coral")
plt.xlabel("Feature index")
plt.ylabel("F-score")
plt.title("Feature Selection: top 5 features highlighted (of 20)")
plt.show()

print(f"Selected {X_selected.shape[1]} features from {X.shape[1]}")
print(f"Top feature indices: {top_features}")

---

## 9. Train-Test Split and Data Leakage

Preprocessing must respect the boundary between training and test data. **Data leakage** occurs when information from the test set influences training, producing optimistically biased evaluation.

**Common leakage mistakes:**

- Computing mean/median on the full dataset (including test) before splitting.
- Fitting the scaler on all data, then splitting.
- Using the target variable to engineer features before splitting.
- Including future information in time-series predictions.

**Correct workflow:**

1. Split data into train and test sets first.
2. Fit preprocessing (imputer, scaler, encoder) on **training data only**.
3. Transform both train and test using the fitted preprocessor.
4. Train the model on preprocessed training data.
5. Evaluate on preprocessed test data.

This ensures the test set simulates truly unseen data — the preprocessor has never seen it during fitting.

In [ ]:
# Demonstrating correct vs leaky preprocessing
X_demo, y_demo = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_demo, y_demo, test_size=0.3, random_state=42)

# CORRECT: fit scaler on train only
scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train)
X_test_scaled = scaler_correct.transform(X_test)  # use train statistics

# WRONG: fit scaler on all data (leakage)
scaler_leaky = StandardScaler()
all_scaled = scaler_leaky.fit_transform(np.vstack([X_train, X_test]))

clf_correct = LogisticRegression().fit(X_train_scaled, y_train)
acc_correct = accuracy_score(y_test, clf_correct.predict(X_test_scaled))

print(f"Correct pipeline test accuracy: {acc_correct:.4f}")
print(f"Train scaler mean (first feature): {scaler_correct.mean_[0]:.4f}")
print(f"Leaky scaler mean (first feature): {scaler_leaky.mean_[0]:.4f}")
print("\nThe leaky scaler's mean differs because it 'saw' test data during fitting.")

---

## 10. Building Preprocessing Pipelines

Manually chaining imputation, encoding, scaling, and modeling is error-prone. Scikit-learn **Pipelines** bundle all steps into a single object that can be fit and evaluated consistently.

A **ColumnTransformer** applies different preprocessing to different column types — numerical columns get imputation and scaling, categorical columns get one-hot encoding — then concatenates the results.

Benefits:
- Prevents data leakage by fitting all steps on training data only.
- Makes the workflow reproducible and deployable.
- Simplifies cross-validation (the entire pipeline is evaluated together).
- One `.fit()` and `.predict()` call handles everything.

In [ ]:
# Full preprocessing pipeline with ColumnTransformer
df_pipe = df.copy()
X_pipe = df_pipe.drop("approved", axis=1)
y_pipe = df_pipe["approved"]

num_features = ["age", "income", "credit_score"]
cat_features = ["employment"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

X_tr, X_te, y_tr, y_te = train_test_split(X_pipe, y_pipe, test_size=0.25, random_state=42)
full_pipeline.fit(X_tr, y_tr)
acc = accuracy_score(y_te, full_pipeline.predict(X_te))

print(f"Pipeline test accuracy: {acc:.4f}")
print(f"\nTransformed feature shape: {full_pipeline.named_steps['preprocessor'].transform(X_tr).shape}")
print("Pipeline steps: impute → scale (num) | impute → one-hot (cat) → logistic regression")

---

## 11. A Practical Preprocessing Checklist

When starting a new machine learning project, work through this sequence:

1. **Load and inspect** — shape, dtypes, missing values, duplicates, basic statistics.
2. **Clean** — fix types, remove duplicates, handle obvious errors.
3. **Split** — separate train and test sets before any transformation.
4. **Handle missing values** — impute or remove based on extent and reason.
5. **Handle outliers** — detect, investigate, cap or remove as appropriate.
6. **Encode categories** — one-hot for nominal, ordinal encoding for ordered categories.
7. **Scale numerical features** — standardize or min-max, fit on train only.
8. **Engineer features** — create domain-driven and interaction features.
9. **Select features** — remove irrelevant or redundant columns.
10. **Build pipeline** — chain all steps for reproducibility.
11. **Validate** — cross-validate the full pipeline, not just the model.

---

## 12. Summary

Data preprocessing and feature engineering transform raw, messy data into the clean numerical representations that machine learning algorithms require. **Missing values** are handled through deletion or imputation. **Outliers** are detected with statistical methods and treated based on whether they are errors or genuine signals. **Categorical variables** are encoded as integers or one-hot vectors. **Feature scaling** ensures comparable magnitudes across numerical inputs.

**Feature engineering** creates new variables from domain knowledge, mathematical transforms, datetime decomposition, and aggregations — often providing more performance gain than switching algorithms. **Feature selection** removes noise and redundancy. **Pipelines** prevent data leakage and make the entire workflow reproducible.

The guiding principle throughout: fit preprocessing on training data only, then apply the learned transformations to test data. This discipline ensures that model evaluation reflects real-world performance on unseen data.